## Kaggle Competition: Irrigation Need Prediction.

In this competition, you need to predict the need of irrigation of the farmer based on the features given to you. 
There are 3 predictions which are low, normal and high. The predictions tell the need of irrigation for the farmer.

## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score
import lightgbm
import optuna

C:\Users\LEGION\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading Data

In [10]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
test_ids = test['id']
print(f'training data shape: {train.shape}')
print(f'test data shape: {test.shape}')

training data shape: (630000, 21)
test data shape: (270000, 20)


## Key findings from EDA

#### 1) There are 10 numerical features and 9 categorical features
#### 2) The target variable is irrigation need which has 3 possible values: low, medium and high. The 57.8% irrigation need is low, 37.9% is medium and only 3.3% is high so there is severe class imbalance in dataset. So we should train our model keeping this in mind,
#### 3) The strong numerical features are: Soil_Moisture, Temperature_C, Rainfall_mm, Wind_Speed_kmh showed clear separation across classes. 
#### 4) All the other weak numerical features consists of: Soil_pH, Organic_Carbon, Electrical_Conductivity, Humidity, Sunlight_Hours, Field_Area_Hectare showed the same variation as target variable irrigation need. 
#### 5) In categorical features all the features behaved the same except Crop_Growth_Stage where two harvesting and sowing showed same variation whereas vegetative and flowering showed same variation so we grouped these 4 into 2 types early growth stage and late growing stage. 

## Checking Null Values

In [12]:
print(train.isnull().sum())
print(test.isnull().sum())

id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
Irrigation_Need            0
dtype: int64
id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
S

## Target Distribution

## Feature Engineering

In [ ]:
# Strong feature combinations
train['heat_wind'] = train['Temperature_C'] * train['Wind_Speed_kmh']
train['water_balance'] = train['Rainfall_mm'] / (train['Temperature_C'] + 1)
train['total_water'] = train['Soil_Moisture'] + train['Rainfall_mm']
train['soil_health'] = train['Soil_pH'] * train['Organic_Carbon']

# From top public notebook
train['Moisture_to_Heat_Ratio'] = train['Soil_Moisture'] / (train['Temperature_C'] + 1)
train['Climate_Stress_Index'] = train['Temperature_C'] * train['Wind_Speed_kmh'] / (train['Humidity'] + 1)
train['Water_Availability_Index'] = (train['Rainfall_mm'] + train['Soil_Moisture']) / (train['Temperature_C'] + 1)

# Groupby encoding
train['Total_Water'] = train['Previous_Irrigation_mm'] + train['Rainfall_mm']
crop_water = train.groupby('Crop_Type')['Total_Water'].mean()
train['avg_water_crop'] = train['Crop_Type'].map(crop_water)

# Categorical grouping
stage_map = {
    'Sowing': 'early_late',
    'Harvesting': 'early_late',
    'Vegetative': 'mid',
    'Flowering': 'mid'
}
train['growth_stage_group'] = train['Crop_Growth_Stage'].map(stage_map)
train['mulching_binary'] = (train['Mulching_Used'] == 'Yes').astype(int)

## Label Encoding

In [ ]:
# Encode target
le_target = LabelEncoder()
train['target_encoded'] = le_target.fit_transform(train['Irrigation_Level'])
print("Target mapping:", dict(zip(le_target.classes_, 
      le_target.transform(le_target.classes_))))

# Encode categorical columns
categorical_cols = train.select_dtypes(include='object').columns.tolist()
categorical_cols = [col for col in categorical_cols if col != 'Irrigation_Level']
print("Encoding:", categorical_cols)

le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])
    le_dict[col] = le


## Model selection and training

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

# This function runs once per trial
# Optuna calls it many times with different parameter suggestions
def objective(trial):
    
    # trial.suggest_int means suggest an integer between these bounds
    # trial.suggest_float means suggest a float between these bounds
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
    }
    
    model = LGBMClassifier(
        **params,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
    
    # Cross validate and return mean score
    score = cross_val_score(
        model, X, y,
        cv=cv,
        scoring='accuracy',
        n_jobs=1
    ).mean()
    
    return score

# Create study — direction maximize means find highest score
study = optuna.create_study(direction='maximize')

# Run 50 trials — each trial is one parameter combination
# n_jobs=-1 means trials run in parallel using all your cores
study.optimize(objective, n_trials=50, n_jobs=-1)

print(f"Best Score: {study.best_value:.4f}")
print(f"Best Params: {study.best_params}")

In [ ]:
final_model = LGBMClassifier(
    **study.best_params,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

final_model.fit(X, y)

## Predicting results and submitting

In [ ]:
predictions = final_model.predict(test)
predictions_labels = le_target.inverse_transform(test_data)

In [ ]:
data = {'id': test_ids, 'Irrigation_Need': predictions_labels}
submission = pd.DataFrame(data)
submission.to_csv('submission_file.csv', index=False)